In [28]:
# We are creating a simple agentic system to stimulate HITL.
# we will ask llm a question, then llm will confrim, is this what you wanna ask? if we approve, the llm will response, otherwise llm will ask us to rephrase the question. This is a simple HITL system.

In [29]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START,END, add_messages
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()
from langgraph.types import interrupt,Command

In [30]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MAIN_LLM=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3,
    api_key=OPENAI_API_KEY,
    max_retries=3
)

FALLBACK_LLM=ChatGoogleGenerativeAI(
    model= 'gemini-1.5-turbo',
    temperature=0.3,
    api_key=GEMINI_API_KEY,
    max_retries=3
)


llm=MAIN_LLM.with_fallbacks([FALLBACK_LLM])

from langchain_core.callbacks import BaseCallbackHandler
class FallbackTracker(BaseCallbackHandler):
    def on_llm_error(self, error: BaseException, **kwargs) -> None:
        print(f"ChatGPT failed with error: {error}. Falling back to Gemini...")

# Pass the callback handler during invoke
tracker = FallbackTracker()

In [31]:
# res=llm.invoke('''Hello, how are you, whats your views on science, in 1 sentence,"
# "config={"callbacks": [tracker]}''')
# print(res.content)
# print("Model that responded:", res.response_metadata)

In [32]:
from typing import Annotated, Literal, Optional, List

class GlobalState(BaseModel):
    messages:Annotated[list, add_messages]=[]
    is_approved:Optional[bool]=Field(default=None, description="Whether the user approved the question or not")
    llm_response:Optional[str]=Field(default=None, description="The response from the LLM after approval")

In [33]:
def chatnode(State:GlobalState)-> dict:
    decision=interrupt(f'''
You asked: {State.messages[-1].content}.
Do you wanna proceed with this question? Please answer with "yes" or "no".    
''')
    if decision.lower() == "yes":
       
        res=llm.invoke(f"{State.messages[-1].content}")
        

        return {
            "is_approved": True,
            "llm_response": res.content
        }

    
    else:
      

        return {
            "is_approved": False,
                "llm_response":" User did not approve the question."
            }

In [34]:
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver()

In [35]:
builder=StateGraph(GlobalState)
builder.add_node("chat_node", chatnode)

builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)

graph=builder.compile(checkpointer=memory)

In [36]:
config = {"configurable": {"thread_id": "session_1"}}

In [37]:
from langchain.messages import HumanMessage


userQ=input("Please enter your question (or type 'exit' to quit): ")
while userQ != "exit":

    inputs = {"messages": [HumanMessage(content=userQ)]}
    for event in graph.stream(inputs,config,stream_mode="updates"):
        print(event)

    human_choice = input("> Proceed? (yes/no): ")

    for event in graph.stream(Command(resume=human_choice),config,stream_mode="updates"):
        if "chat_node" in event:
            print(f"AI: {event['chat_node']['llm_response']}\n")
        

    userQ = input("Please enter your question (or type 'exit' to quit): ")


{'__interrupt__': (Interrupt(value='\nYou asked: do you know me?.\nDo you wanna proceed with this question? Please answer with "yes" or "no".    \n', id='3cb905e212cf32ac8e71ea3838431943'),)}
AI:  User did not approve the question.

{'__interrupt__': (Interrupt(value='\nYou asked: do you know ratan tata?.\nDo you wanna proceed with this question? Please answer with "yes" or "no".    \n', id='8d6c488ca4c2fb8320bb66549318f11d'),)}
AI: Yes, Ratan Tata is an Indian industrialist and the former chairman of Tata Sons, the holding company of the Tata Group, one of India's largest and oldest conglomerates. He was born on December 28, 1937, and has played a significant role in expanding the Tata Group's global presence and diversifying its business interests. Under his leadership, the Tata Group acquired several international companies, including Jaguar Land Rover and Corus Steel. Ratan Tata is also known for his philanthropic efforts and has been involved in various social initiatives. He is w